# Finetuning di DeepLabV3 su CamVid
Lo scopo di questo notebook è quello di fare finetuning del modello pre-addestrato DeepLabV3 con backbone ResNet50 sul dataset CamVid. Per poter effettuare il retraining di questa rete viene sfruttato Kaggle ma molti dei file che si trovano in questo notebook sono dei moduli presenti anche sulla repository GitHub del progetto.

# Configurazione
Scrivo un file `conf.py` in cui contengo tutte le costanti del progetto. 

In [ ]:
%%writefile /kaggle/working/conf.py
import numpy as np
from pathlib import Path

# ---------------------------------------------------------------------------
# Percorsi
# ---------------------------------------------------------------------------
DATA_ROOT = Path('/kaggle/input/datasets/carlolepelaars/camvid/CamVid')
OUTPUT_DIR = Path('/kaggle/working/output')

# ---------------------------------------------------------------------------
# Modello
# ---------------------------------------------------------------------------
MODEL = "deeplabv3"
LOSS  = "ce"

# ---------------------------------------------------------------------------
# CamVid: 11 classi semantiche + Void (ignorato)
# ---------------------------------------------------------------------------
N_CLASSES = 11

CLASS_NAMES = [
    "Sky",
    "Building",
    "Pole",
    "Road",
    "Sidewalk",
    "Tree",
    "SignSymbol",
    "Fence",
    "Car",
    "Pedestrian",
    "Bicyclist",
]

CAMVID_PALETTE = np.array([
    [128, 128, 128],   #  0 Sky
    [128,   0,   0],   #  1 Building
    [192, 192, 128],   #  2 Pole
    [128,  64, 128],   #  3 Road
    [  0,   0, 192],   #  4 Sidewalk
    [128, 128,   0],   #  5 Tree
    [192, 128, 128],   #  6 SignSymbol
    [ 64,  64, 128],   #  7 Fence
    [ 64,   0, 128],   #  8 Car
    [ 64,  64,   0],   #  9 Pedestrian
    [  0, 128, 192],   # 10 Bicyclist
], dtype=np.uint8)

# Palette RGB ufficiale CamVid (una riga per classe, stesso ordine di CLASS_NAMES)
# Mapping 32 classi CamVid -> 11 classi (Void (0,0,0) resta escluso -> IGNORE_INDEX)
CAMVID_32_TO_11_RGB = {
    # 0: Sky
    (128, 128, 128): 0,  # Sky
    # 1: Building
    (128,   0,   0): 1,  # Building
    (  0, 128,  64): 1,  # Bridge
    ( 64, 192,   0): 1,  # Wall
    ( 64,   0,  64): 1,  # Tunnel
    (192,   0, 128): 1,  # Archway
    # 2: Pole
    (192, 192, 128): 2,  # Column_Pole
    (  0,   0,  64): 2,  # TrafficCone
    # 3: Road
    (128,  64, 128): 3,  # Road
    (128,   0, 192): 3,  # LaneMkgsDriv
    (192,   0,  64): 3,  # LaneMkgsNonDriv
    # 4: Sidewalk
    (  0,   0, 192): 4,  # Sidewalk
    ( 64, 192, 128): 4,  # ParkingBlock
    (128, 128, 192): 4,  # RoadShoulder
    # 5: Tree
    (128, 128,   0): 5,  # Tree
    (192, 192,   0): 5,  # VegetationMisc
    # 6: SignSymbol
    (192, 128, 128): 6,  # SignSymbol
    (128, 128,  64): 6,  # Misc_Text
    (  0,  64,  64): 6,  # TrafficLight
    # 7: Fence
    ( 64,  64, 128): 7,  # Fence
    # 8: Car
    ( 64,   0, 128): 8,  # Car
    ( 64, 128, 192): 8,  # SUVPickupTruck
    (192, 128, 192): 8,  # Truck_Bus
    (192,  64, 128): 8,  # Train
    (128,  64,  64): 8,  # OtherMoving
    # 9: Pedestrian
    ( 64,  64,   0): 9,  # Pedestrian
    (192, 128,  64): 9,  # Child
    ( 64,   0, 192): 9,  # CartLuggagePram
    ( 64, 128,  64): 9,  # Animal
    # 10: Bicyclist
    (  0, 128, 192): 10, # Bicyclist
    (192,   0, 192): 10, # MotorcycleScooter
}

# Pixel Void (0, 0, 0) → ignorato durante training e valutazione
IGNORE_INDEX = 255

# ---------------------------------------------------------------------------
# Immagini – CamVid nativo: 960 × 720;  si ridimensiona a 480 × 360
# ---------------------------------------------------------------------------
IMG_HEIGHT = 720
IMG_WIDTH  = 960

# Statistiche ImageNet (backbone ResNet pre-addestrato su ImageNet)
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)

# ---------------------------------------------------------------------------
# Training
# ---------------------------------------------------------------------------
BATCH_SIZE  = 8
SEED        = 42
SCALE_RANGE = (0.75, 1.5)   # zoom casuale durante il training

LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
AUX_WEIGHT   = 0.4

# File di utils
Mapping su Kaggle del file utils.py

In [ ]:
%%writefile /kaggle/working/utils.py
import numpy as np
import torch

import conf

# ---------------------------------------------------------------------------
# Lookup table RGB → indice di classe
# ---------------------------------------------------------------------------
# Costruiamo una LUT 256×256×256 → uint8.
# Tutti i pixel non riconosciuti ottengono IGNORE_INDEX (255 = Void).
_RGB_TO_CLASS = np.full((256, 256, 256), conf.IGNORE_INDEX, dtype=np.uint8)
for (r, g, b), class_idx in conf.CAMVID_32_TO_11_RGB.items():
    _RGB_TO_CLASS[r, g, b] = class_idx


def encode_mask(mask_rgb: np.ndarray) -> np.ndarray:
    """Converte una maschera RGB [H, W, 3] in una maschera di indici [H, W].

    I pixel con colore non in palette diventano IGNORE_INDEX (255).
    """
    return _RGB_TO_CLASS[mask_rgb[..., 0], mask_rgb[..., 1], mask_rgb[..., 2]]


def decode_segmap(mask: np.ndarray) -> np.ndarray:
    """Converte una maschera di indici [H, W] in un'immagine RGB [H, W, 3]."""
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for class_idx, color in enumerate(conf.CAMVID_PALETTE):
        rgb[mask == class_idx] = color
    return rgb


# ---------------------------------------------------------------------------
# Utility generali
# ---------------------------------------------------------------------------
def seed_everything(seed: int) -> None:
    """Fissa i seed di random, numpy e torch per la riproducibilità."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# File di tranformations
Mapping del file transformations.py

In [ ]:
%%writefile /kaggle/working/transforms.py
import albumentations as A
from albumentations.pytorch import ToTensorV2

import conf


def get_train_transforms(
    mean: tuple[float, ...] = conf.IMAGENET_MEAN,
    std:  tuple[float, ...] = conf.IMAGENET_STD,
    height: int = conf.IMG_HEIGHT,
    width:  int = conf.IMG_WIDTH,
    crop_size: tuple[int, int] | None = None,
    hflip: bool = False,
    scale_range: tuple[float, float] | None = None,
    color_jitter: bool = False,
) -> A.Compose:
    """Trasformazioni di train (tutte opzionali tranne resize e normalizzazione).

    Ordine fisso:
    1. Resize a height × width
    2. RandomScale  (se scale_range è fornito)
    3. RandomCrop   (se crop_size è fornito; padding con IGNORE_INDEX sulla maschera)
    4. HorizontalFlip (se hflip=True, p=0.5)
    5. ColorJitter  (se color_jitter=True, p=0.5)
    6. Normalize + ToTensorV2
    """
    ops = [A.Resize(height=height, width=width)]

    if scale_range is not None:
        ops.append(A.RandomScale(
            scale_limit=(scale_range[0] - 1, scale_range[1] - 1),
            p=1.0,
        ))

    if crop_size is not None:
        ops.append(A.RandomCrop(
            *crop_size,
            pad_if_needed=True,
            pad_position="random",
            fill=0,
            fill_mask=conf.IGNORE_INDEX,
        ))

    if hflip:
        ops.append(A.HorizontalFlip(p=0.5))

    if color_jitter:
        ops.append(A.ColorJitter(
            brightness=0.2, contrast=0.2, saturation=0.2, hue=0.0, p=0.5,
        ))

    ops += [A.Normalize(mean=mean, std=std), ToTensorV2()]
    return A.Compose(ops)


def get_val_transforms(
    mean: tuple[float, ...] = conf.IMAGENET_MEAN,
    std:  tuple[float, ...] = conf.IMAGENET_STD,
    height: int = conf.IMG_HEIGHT,
    width:  int = conf.IMG_WIDTH,
) -> A.Compose:
    """Trasformazioni deterministiche per validation e test (nessuna augmentation)."""
    return A.Compose([
        A.Resize(height=height, width=width),
        A.Normalize(mean=mean, std=std),
        ToTensorV2(),
    ])


# Notebook del file dataset
Mapping del file dataset.ipynb

# Dataset CamVid per DeepLabV3

Questo notebook definisce:
- **`CamVidDataset`** – classe `Dataset` PyTorch per il caricamento di immagini e maschere CamVid
- **`make_loader`** – funzione che costruisce un `DataLoader` con le impostazioni standard
- **`build_dataloaders`** – funzione di alto livello che restituisce i tre loader (train, val, test)

Struttura della cartella `archive/`:
```
archive/
├── train/           # immagini  → 0001TP_009210.png
├── train_labels/    # maschere  → 0001TP_009210_L.png  (suffisso _L)
├── val/
├── val_labels/
├── test/
├── test_labels/
└── class_dict.csv
```

## 1. Import

In [ ]:
import os
from pathlib import Path

import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader

import conf
from transforms import get_train_transforms, get_val_transforms
from utils import encode_mask

## 2. Classe `CamVidDataset`

In [ ]:
class CamVidDataset(Dataset):
    """Dataset PyTorch per CamVid.

    Parametri
    ----------
    images : list[Path]
        Percorsi delle immagini RGB.
    masks : list[Path]
        Percorsi delle maschere RGB (stessa lunghezza di `images`).
    transforms : A.Compose | None
        Pipeline albumentations applicata a immagine e maschera insieme.
        Se None, viene restituito il tensore grezzo normalizzato in [0, 1].
    """

    def __init__(self, images: list[Path], masks: list[Path], transforms=None):
        super().__init__()
        assert len(images) == len(masks), "Il numero di immagini e maschere deve coincidere."
        self.images = images
        self.masks = masks
        self.transforms = transforms

    def __len__(self) -> int:
        return len(self.images)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        """Restituisce (immagine, maschera) come tensori PyTorch.

        - immagine : [3, H, W]  float32, normalizzata con media/std ImageNet
        - maschera : [H, W]     int64,   valori in [0, N_CLASSES-1] o IGNORE_INDEX
        """
        img  = np.array(Image.open(self.images[index]).convert("RGB"))
        mask = encode_mask(np.array(Image.open(self.masks[index]).convert("RGB")))

        if self.transforms is not None:
            out = self.transforms(image=img, mask=mask)
            return out["image"], out["mask"].long()

        # Fallback senza trasformazioni
        img_tensor  = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0
        mask_tensor = torch.from_numpy(mask).long()
        return img_tensor, mask_tensor

## 3. Raccolta dei percorsi per split

In CamVid ogni maschera ha lo stesso nome dell'immagine con il suffisso `_L` aggiunto prima dell'estensione  
(es. `0001TP_009210.png` → `0001TP_009210_L.png`).

In [ ]:
def collect_split(img_dir: Path, mask_dir: Path) -> tuple[list[Path], list[Path]]:
    """Legge tutti i file PNG in `img_dir` e individua la maschera corrispondente in `mask_dir`.

    La maschera si ottiene aggiungendo il suffisso `_L` al nome dell'immagine:
        0001TP_009210.png  →  0001TP_009210_L.png
    """
    images = sorted(img_dir.glob("*.png"))
    masks  = [mask_dir / (img.stem + "_L.png") for img in images]

    missing = [m for m in masks if not m.exists()]
    assert not missing, f"Maschere mancanti in {mask_dir}: {missing}"

    return images, masks

## 4. `make_loader`

In [ ]:
def make_loader(dataset: Dataset, batch_size: int, shuffle: bool) -> DataLoader:
    """Costruisce un DataLoader con impostazioni ottimizzate.

    - `drop_last=True` solo in training per evitare batch di dimensione 1.
    - `persistent_workers=True` se num_workers > 0 (riduce l'overhead di spawn).
    """
    num_workers = min(4, os.cpu_count() or 1)
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=shuffle,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=num_workers > 0,
        generator=torch.Generator().manual_seed(conf.SEED),
    )

## 5. `build_dataloaders`

In [ ]:
def build_dataloaders(
    root: Path = conf.DATA_ROOT,
    batch_size: int = conf.BATCH_SIZE,
    height: int = conf.IMG_HEIGHT,
    width: int = conf.IMG_WIDTH,
    augment: bool = False,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    """Restituisce i DataLoader di train, val e test.

    Parametri
    ----------
    root : Path
        Cartella radice del dataset CamVid (di default `archive/`).
    batch_size : int
        Dimensione del batch.
    height, width : int
        Dimensioni a cui ridimensionare ogni immagine.
    augment : bool
        Se True, applica data augmentation al training
        (zoom casuale, crop casuale, flip orizzontale, color jitter).
    """
    # --- raccolta percorsi ---
    train_imgs, train_masks = collect_split(root / "train",  root / "train_labels")
    val_imgs,   val_masks   = collect_split(root / "val",    root / "val_labels")
    test_imgs,  test_masks  = collect_split(root / "test",   root / "test_labels")

    # --- trasformazioni ---
    if augment:
        train_tf = get_train_transforms(
            height=height, width=width,
            crop_size=(height, width),
            hflip=True,
            scale_range=conf.SCALE_RANGE,
            color_jitter=True,
        )
    else:
        train_tf = get_train_transforms(height=height, width=width)

    val_tf = get_val_transforms(height=height, width=width)

    # --- dataset ---
    train_ds = CamVidDataset(train_imgs, train_masks, transforms=train_tf)
    val_ds   = CamVidDataset(val_imgs,   val_masks,   transforms=val_tf)
    test_ds  = CamVidDataset(test_imgs,  test_masks,  transforms=val_tf)

    # --- loader ---
    train_loader = make_loader(train_ds, batch_size, shuffle=True)
    val_loader   = make_loader(val_ds,   batch_size, shuffle=False)
    test_loader  = make_loader(test_ds,  batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

## 6. Visualizzazione campione casuale

In [ ]:
import random
import matplotlib.pyplot as plt
from utils import decode_segmap

# Campiona un'immagine casuale dallo split di train
train_imgs, train_masks = collect_split(conf.DATA_ROOT / "train", conf.DATA_ROOT / "train_labels")
idx = random.randrange(len(train_imgs))

img      = np.array(Image.open(train_imgs[idx]).convert("RGB"))
mask     = encode_mask(np.array(Image.open(train_masks[idx]).convert("RGB")))
mask_rgb = decode_segmap(mask)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].imshow(img)
axes[0].set_title(f"Immagine  [{train_imgs[idx].name}]")
axes[0].axis("off")

im = axes[1].imshow(mask, cmap="tab20", vmin=0, vmax=conf.N_CLASSES - 1)
axes[1].set_title(f"Maschera indici  [{train_masks[idx].name}]")
axes[1].axis("off")
plt.colorbar(im, ax=axes[1], ticks=range(conf.N_CLASSES), shrink=0.8)

axes[2].imshow(mask_rgb)
axes[2].set_title(f"Maschera RGB  [{train_masks[idx].name}]")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 7. Verifica rapida

In [ ]:
train_loader, val_loader, test_loader = build_dataloaders(augment=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val   batches : {len(val_loader)}")
print(f"Test  batches : {len(test_loader)}")

imgs, masks = next(iter(train_loader))
print(f"\nBatch immagini : {imgs.shape}   dtype={imgs.dtype}")
print(f"Batch maschere : {masks.shape}  dtype={masks.dtype}")
print(f"Classi uniche nel batch: {masks.unique().tolist()}")

# modello (models.py)

In [ ]:
%%writefile /kaggle/working/models.py
import torch
import torch.nn as nn
from torchvision.models.segmentation import (
    deeplabv3_resnet50,
    DeepLabV3_ResNet50_Weights,
)
import conf

# Caricamento DeepLabV3-ResNet50 pre-addestrato con auxiliary head abilitata
model = deeplabv3_resnet50(
    weights=DeepLabV3_ResNet50_Weights.DEFAULT,
    aux_loss=True,
)

# 1. Scongela tutta la rete (backbone + main head + aux head)
for param in model.parameters():
    param.requires_grad = True

# 2. Sostituisce l'ultimo Conv2d 1x1 della head principale (ASPP / DeepLabHead)
in_ch_main = model.classifier[4].in_channels  # 256
model.classifier[4] = nn.Conv2d(in_ch_main, conf.N_CLASSES, kernel_size=1)

# 3. Sostituisce l'ultimo Conv2d 1x1 della head ausiliaria (FCNHead su layer3)
in_ch_aux = model.aux_classifier[4].in_channels  # 256
model.aux_classifier[4] = nn.Conv2d(in_ch_aux, conf.N_CLASSES, kernel_size=1)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 4. Differential Learning Rate: backbone (LR basso) vs head principale + ausiliaria (LR 10x)
backbone_params = list(model.backbone.parameters())
head_params = (
    list(model.classifier.parameters())
    + list(model.aux_classifier.parameters())
)
assert len(backbone_params) + len(head_params) == len(list(model.parameters())), (
    "Parametri del modello non coperti dai param_groups dell'ottimizzatore."
)

optimizer = torch.optim.AdamW(
    [
        {"params": backbone_params, "lr": conf.LR_BACKBONE},
        {"params": head_params, "lr": conf.LR_HEAD},
    ],
    weight_decay=conf.WEIGHT_DECAY,
)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(
    f"Parametri addestrabili: {n_trainable:,} / {n_total:,} "
    f"({100 * n_trainable / n_total:.2f}%)"
)

## metriche
dopo aver fatto fine tuning faccio un file 'metrics.py' in cui sostanzialmente determino la loss da usare (inizialmente ce poi da vedere anche qui se usare una dice loss o altre), **la loss viene settata in conf.py e poi implementata in metrics.py** e poi setto delle metriche di valutazione delle performance del modello

In [ ]:
%%writefile /kaggle/working/metrics.py
"""Metriche di segmentazione calcolate a partire dalla matrice di confusione.

Convenzione: cm[t, p] = numero di pixel con classe vera t predetti come p.
I pixel con ignore_index sono esclusi prima di costruire la matrice.
"""
import torch


def confusion_matrix(
    preds: torch.Tensor,
    targets: torch.Tensor,
    n_classes: int,
    ignore_index: int = 255,
) -> torch.Tensor:
    """Matrice di confusione [n_classes, n_classes] (int64) da predizioni e target della stessa shape."""
    valid = targets != ignore_index
    t = targets[valid].long()
    p = preds[valid].long()
    # Ogni coppia (t, p) diventa un indice unico t*C + p: un solo bincount conta tutte le coppie
    counts = torch.bincount(t * n_classes + p, minlength=n_classes * n_classes)
    return counts.reshape(n_classes, n_classes)


def _tp_fp_fn(cm: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    cm = cm.double()
    tp = cm.diag()
    fp = cm.sum(dim=0) - tp  # predetti come c ma di un'altra classe
    fn = cm.sum(dim=1) - tp  # di classe c ma predetti come altro
    return tp, fp, fn


def iou_per_class(cm: torch.Tensor) -> torch.Tensor:
    """IoU = TP / (TP + FP + FN). NaN per le classi assenti sia nel target sia nelle predizioni."""
    tp, fp, fn = _tp_fp_fn(cm)
    return tp / (tp + fp + fn)  # 0/0 -> NaN: classe da non contare nella media


def f1_per_class(cm: torch.Tensor) -> torch.Tensor:
    """F1 (= Dice) = 2TP / (2TP + FP + FN). NaN per le classi assenti."""
    tp, fp, fn = _tp_fp_fn(cm)
    return 2 * tp / (2 * tp + fp + fn)


def mean_iou(cm: torch.Tensor) -> float:
    """Media dell'IoU sulle sole classi presenti (ignora i NaN)."""
    return torch.nanmean(iou_per_class(cm)).item()


def mean_f1(cm: torch.Tensor) -> float:
    return torch.nanmean(f1_per_class(cm)).item()


def pixel_accuracy(cm: torch.Tensor) -> float:
    """Frazione di pixel validi classificati correttamente."""
    total = cm.sum()
    return (cm.diag().sum() / total).item() if total > 0 else float("nan")


In [ ]:
%%writefile /kaggle/working/train_utils.py
import torch
import torch.nn as nn
import conf
import metrics
import numpy as np


# ---------------------------------------------------------------------------
# Loss
# ---------------------------------------------------------------------------
def get_criterion(
    loss_name: str = conf.LOSS,
    weight: torch.Tensor | None = None,
) -> nn.Module:
    """Restituisce la funzione di loss configurata con ignore_index."""
    if loss_name == "ce":
        return nn.CrossEntropyLoss(weight=weight, ignore_index=conf.IGNORE_INDEX)
    raise ValueError(f"Loss non supportata: {loss_name}")


def compute_loss(
    outputs: dict[str, torch.Tensor] | torch.Tensor,
    targets: torch.Tensor,
    criterion: nn.Module,
    aux_weight: float = 0.4,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Restituisce (total_loss, main_loss) per backprop e logging omogeneo."""
    if isinstance(outputs, dict):
        main_loss = criterion(outputs["out"], targets)
        total_loss = main_loss
        if "aux" in outputs and outputs["aux"] is not None:
            total_loss = total_loss + aux_weight * criterion(outputs["aux"], targets)
        return total_loss, main_loss
    loss = criterion(outputs, targets)
    return loss, loss


# ---------------------------------------------------------------------------
# Accumulatore di metriche per epoca (wrapper su metrics.py)
# ---------------------------------------------------------------------------
class SegmentationMeter:
    """Accumula la confusion matrix su un intero split delegando il calcolo a metrics.py."""

    def __init__(
        self,
        num_classes: int = conf.N_CLASSES,
        ignore_index: int = conf.IGNORE_INDEX,
        class_names: list[str] = conf.CLASS_NAMES,
    ):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.class_names = class_names
        self.cm: torch.Tensor | None = None

    def reset(self) -> None:
        self.cm = None

    @torch.no_grad()
    def update(self, preds: torch.Tensor, targets: torch.Tensor) -> None:
        """Accetta logit [B, C, H, W] o indici [B, H, W] e accumula la confusion matrix."""
        if preds.ndim == 4:
            preds = preds.argmax(dim=1)

        batch_cm = metrics.confusion_matrix(
            preds=preds,
            targets=targets,
            n_classes=self.num_classes,
            ignore_index=self.ignore_index,
        )
        self.cm = batch_cm if self.cm is None else self.cm + batch_cm

    def compute(self) -> dict:
        """Restituisce tutte le metriche esposte da metrics.py e la confusion matrix."""
        if self.cm is None or self.cm.sum() == 0:
            nan_dict = {name: float("nan") for name in self.class_names}
            return {
                "miou": float("nan"),
                "mf1": float("nan"),
                "pixel_acc": float("nan"),
                "iou_per_class": nan_dict,
                "f1_per_class": nan_dict,
                "confusion_matrix": np.zeros((self.num_classes, self.num_classes), dtype=np.int64),
            }

        iou_vals = metrics.iou_per_class(self.cm).tolist()
        f1_vals = metrics.f1_per_class(self.cm).tolist()

        return {
            "miou": metrics.mean_iou(self.cm),
            "mf1": metrics.mean_f1(self.cm),
            "pixel_acc": metrics.pixel_accuracy(self.cm),
            "iou_per_class": dict(zip(self.class_names, iou_vals)),
            "f1_per_class": dict(zip(self.class_names, f1_vals)),
            "confusion_matrix": self.cm.detach().cpu().numpy(),
        }


## parte di training
quindi ho lasciato la rete unfreezed end-to-end, con AdamW per aggiornare i pesi ora si passa alla parte di training della rete vera e propria

In [ ]:
%%writefile /kaggle/working/train.py
from pathlib import Path
import time

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import conf
from train_utils import compute_loss, SegmentationMeter


def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: nn.Module,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    aux_weight: float = conf.AUX_WEIGHT,
) -> tuple[float, dict]:
    model.train()

    meter = SegmentationMeter()
    running_loss = 0.0
    use_amp = device.type == "cuda"

    for imgs, masks in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(imgs)
            # Backprop su total_loss (main + aux_weight * aux); logging su main_loss
            total_loss, main_loss = compute_loss(
                outputs, masks, criterion, aux_weight=aux_weight
            )

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += main_loss.item()
        meter.update(outputs["out"].detach(), masks)

    avg_loss = running_loss / len(loader)
    return avg_loss, meter.compute()


@torch.no_grad()
def validate_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple[float, dict]:
    model.eval()
    meter = SegmentationMeter()
    running_loss = 0.0
    use_amp = device.type == "cuda"

    for imgs, masks in loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(imgs)
            # In validation si valuta solo la head principale ('out')
            loss = criterion(outputs["out"], masks)

        running_loss += loss.item()
        meter.update(outputs["out"], masks)

    avg_loss = running_loss / len(loader)
    return avg_loss, meter.compute()


def fit(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    scheduler: torch.optim.lr_scheduler.LRScheduler | None,
    criterion: nn.Module,
    device: torch.device,
    num_epochs: int = 100,
    patience: int = 10,
    save_dir: Path = conf.OUTPUT_DIR,
    checkpoint_name: str = "best_model.pth",
    aux_weight: float = conf.AUX_WEIGHT,
) -> dict:
    save_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = save_dir / checkpoint_name

    scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

    history = {
        "train_loss": [], "val_loss": [],
        "train_miou": [], "val_miou": [],
        "val_mf1": [], "val_pixel_acc": [],
    }

    best_miou = -1.0
    epochs_no_improve = 0

    for epoch in range(1, num_epochs + 1):
        t0 = time.perf_counter()

        train_loss, train_m = train_one_epoch(
            model, train_loader, optimizer, criterion, scaler, device, aux_weight
        )
        val_loss, val_m = validate_one_epoch(
            model, val_loader, criterion, device
        )

        if scheduler is not None:
            scheduler.step()

        elapsed = time.perf_counter() - t0

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_miou"].append(train_m["miou"])
        history["val_miou"].append(val_m["miou"])
        history["val_mf1"].append(val_m["mf1"])
        history["val_pixel_acc"].append(val_m["pixel_acc"])

        improved = val_m["miou"] > best_miou
        if improved:
            best_miou = val_m["miou"]
            epochs_no_improve = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_miou": best_miou,
                "val_metrics": val_m,
            }, ckpt_path)
        else:
            epochs_no_improve += 1

        lr_bb = optimizer.param_groups[0]["lr"]
        lr_head = optimizer.param_groups[1]["lr"] if len(optimizer.param_groups) > 1 else lr_bb
        flag = " *" if improved else ""
        print(
            f"Ep {epoch:02d}/{num_epochs:02d} [{elapsed:4.1f}s] | "
            f"Tr Loss: {train_loss:.4f} mIoU: {train_m['miou']:.4f} | "
            f"Val Loss: {val_loss:.4f} mIoU: {val_m['miou']:.4f} "
            f"F1: {val_m['mf1']:.4f} Acc: {val_m['pixel_acc']:.4f} | "
            f"LR(bb/hd): {lr_bb:.2e}/{lr_head:.2e}{flag}"
        )

        if epochs_no_improve >= patience:
            print(f"Early stopping attivato all'epoca {epoch} (best val mIoU: {best_miou:.4f}).")
            break

    return history

In [ ]:
import matplotlib.pyplot as plt
import torch
from models import model, optimizer, device

import conf
from utils import seed_everything
from train_utils import get_criterion
from train import fit

NUM_EPOCHS = 100
PATIENCE = 15  
criterion = get_criterion(conf.LOSS)

# Snapshot su CPU di pesi iniziali e LR per riutilizzare model e optimizer gia' definiti
init_weights = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
init_lrs = [pg["lr"] for pg in optimizer.param_groups]

histories = {}

for name, augment in [("baseline", False), ("augmented", True)]:
    print(f"\n=== {name.upper()} (augment={augment}) ===")
    seed_everything(conf.SEED)

    model.load_state_dict(init_weights)
    optimizer.state.clear()
    for pg, lr in zip(optimizer.param_groups, init_lrs):
        pg["lr"] = lr
        pg.pop("initial_lr", None)

    train_loader, val_loader, test_loader = build_dataloaders(augment=augment)
    scheduler = torch.optim.lr_scheduler.PolynomialLR(
        optimizer, total_iters=NUM_EPOCHS, power=0.9
    )

    histories[name] = fit(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        criterion=criterion,
        device=device,
        num_epochs=NUM_EPOCHS,
        patience=PATIENCE,
        checkpoint_name=f"best_{name}.pth",
    )

# Plot sovrapposto Training Loss vs Validation Loss
fig, axes = plt.subplots(1, len(histories), figsize=(14, 5), sharey=True)
if len(histories) == 1:
    axes = [axes]

for ax, (name, hist) in zip(axes, histories.items()):
    epochs = range(1, len(hist["train_loss"]) + 1)
    ax.plot(epochs, hist["train_loss"], label="Training Loss", linewidth=2)
    ax.plot(epochs, hist["val_loss"], label="Validation Loss", linewidth=2, linestyle="--")
    ax.set_title(f"{name.capitalize()} (Best Val mIoU: {max(hist['val_miou']):.4f})")
    ax.set_xlabel("Epoca")
    ax.set_ylabel("Cross-Entropy Loss")
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
loss_plot_path = conf.OUTPUT_DIR / "loss_curves.png"
plt.savefig(loss_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Grafico loss salvato in: {loss_plot_path}")

## parte di testing

In [ ]:
%%writefile /kaggle/working/eval_test.py
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import conf
from utils import decode_segmap
from train_utils import SegmentationMeter


@torch.no_grad()
def evaluate_test(
    model: nn.Module,
    test_loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    checkpoint_path: Path,
) -> tuple[float, dict]:
    """Carica il miglior checkpoint e valuta il modello sullo split di test."""
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()

    meter = SegmentationMeter()
    running_loss = 0.0
    use_amp = device.type == "cuda"

    for imgs, masks in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(imgs)
            loss = criterion(outputs["out"], masks)

        running_loss += loss.item()
        meter.update(outputs["out"], masks)

    avg_loss = running_loss / len(test_loader)
    results = meter.compute()

    # Tabella metriche per classe e globali
    print(f"\n=== Risultati Test Set [{checkpoint_path.name}] (Epoca ckpt: {ckpt.get('epoch', '?')}) ===")
    print(f"{'Classe':<15} | {'IoU':>8} | {'F1 / Dice':>9}")
    print("-" * 38)
    for cls_name in conf.CLASS_NAMES:
        iou = results["iou_per_class"][cls_name]
        f1 = results["f1_per_class"][cls_name]
        print(f"{cls_name:<15} | {iou:8.4f} | {f1:9.4f}")
    print("-" * 38)
    print(f"Test Loss      : {avg_loss:.4f}")
    print(f"mIoU           : {results['miou']:.4f}")
    print(f"mF1 (Dice)     : {results['mf1']:.4f}")
    print(f"Pixel Accuracy : {results['pixel_acc']:.4f}")

    return avg_loss, results


def _denormalize(img_tensor: torch.Tensor) -> np.ndarray:
    mean = np.array(conf.IMAGENET_MEAN, dtype=np.float32).reshape(1, 1, 3)
    std = np.array(conf.IMAGENET_STD, dtype=np.float32).reshape(1, 1, 3)
    img = img_tensor.detach().cpu().permute(1, 2, 0).numpy()
    return np.clip(img * std + mean, 0.0, 1.0)


def _make_error_map(pred: np.ndarray, gt: np.ndarray) -> np.ndarray:
    """Nero = corretto, Rosso = errore, Grigio = Void (IGNORE_INDEX)."""
    err_rgb = np.zeros((*gt.shape, 3), dtype=np.uint8)
    void_mask = gt == conf.IGNORE_INDEX
    wrong_mask = (~void_mask) & (pred != gt)
    err_rgb[wrong_mask] = [220, 30, 30]
    err_rgb[void_mask] = [120, 120, 120]
    return err_rgb


@torch.no_grad()
def plot_qualitative_results(
    model: nn.Module,
    test_loader: DataLoader,
    device: torch.device,
    num_samples: int = 4,
    save_path: Path | None = None,
) -> None:
    """Mostra affiancati: Immagine, Ground Truth, Predizione e Mappa degli errori."""
    model.eval()
    imgs, masks = next(iter(test_loader))
    n = min(num_samples, imgs.size(0))

    imgs_dev = imgs[:n].to(device)
    with torch.amp.autocast(device_type=device.type, enabled=(device.type == "cuda")):
        preds = model(imgs_dev)["out"].argmax(dim=1).cpu().numpy()

    fig, axes = plt.subplots(n, 4, figsize=(20, 4 * n))
    if n == 1:
        axes = axes[None, :]

    titles = ["Immagine", "Ground Truth", "Predizione", "Errori (Rosso=Err, Grigio=Void)"]
    for col, title in enumerate(titles):
        axes[0, col].set_title(title, fontsize=12)

    for i in range(n):
        gt_np = masks[i].numpy()
        pred_np = preds[i]

        axes[i, 0].imshow(_denormalize(imgs[i]))
        axes[i, 1].imshow(decode_segmap(gt_np))
        axes[i, 2].imshow(decode_segmap(pred_np))
        axes[i, 3].imshow(_make_error_map(pred_np, gt_np))

        for col in range(4):
            axes[i, col].axis("off")

    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Analisi qualitativa salvata in: {save_path}")
    plt.show()

def plot_confusion_matrices(
    cm_dict: dict[str, np.ndarray],
    class_names: list[str] = conf.CLASS_NAMES,
    normalize: bool = True,
    save_path: Path | None = None,
) -> None:
    """Visualizza affiancate le confusion matrix (es. baseline vs augmented)."""
    n_models = len(cm_dict)
    fig, axes = plt.subplots(1, n_models, figsize=(9 * n_models, 8))
    if n_models == 1:
        axes = [axes]

    for ax, (name, cm) in zip(axes, cm_dict.items()):
        if normalize:
            row_sums = cm.sum(axis=1, keepdims=True)
            mat = np.divide(
                cm.astype(np.float64),
                row_sums,
                out=np.zeros_like(cm, dtype=np.float64),
                where=row_sums != 0,
            )
            fmt, vmax = ".2f", 1.0
        else:
            mat, fmt, vmax = cm, "d", None

        im = ax.imshow(
            mat,
            interpolation="nearest",
            cmap="Blues",
            vmin=0.0 if normalize else None,
            vmax=vmax,
        )
        suffix = " (Norm. per riga)" if normalize else ""
        ax.set_title(f"Confusion Matrix - {name.capitalize()}{suffix}", fontsize=12)
        ax.set_xticks(range(len(class_names)))
        ax.set_yticks(range(len(class_names)))
        ax.set_xticklabels(class_names, rotation=45, ha="right")
        ax.set_yticklabels(class_names)
        ax.set_xlabel("Classe Predetta")
        ax.set_ylabel("Classe Reale")

        thresh = (mat.max() / 2.0) if mat.size > 0 else 0.5
        for i in range(mat.shape[0]):
            for j in range(mat.shape[1]):
                ax.text(
                    j, i, format(mat[i, j], fmt),
                    ha="center", va="center", fontsize=8,
                    color="white" if mat[i, j] > thresh else "black",
                )
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    plt.tight_layout()
    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Confusion matrices salvate in: {save_path}")
    plt.show()

In [ ]:
from eval_test import evaluate_test, plot_qualitative_results, plot_confusion_matrices

test_results = {}
confusion_matrices = {}

# Valuta sul test set sia il modello senza augmentation (baseline) sia quello con (augmented)
for exp_name in ["baseline", "augmented"]:
    ckpt_path = conf.OUTPUT_DIR / f"best_{exp_name}.pth"
    loss, metrics_res = evaluate_test(
        model=model,
        test_loader=test_loader,
        criterion=criterion,
        device=device,
        checkpoint_path=ckpt_path,
    )
    test_results[exp_name] = (loss, metrics_res)
    confusion_matrices[exp_name] = metrics_res["confusion_matrix"]

# Plot affiancato delle confusion matrix normalizzate per riga (recall per classe)
plot_confusion_matrices(
    cm_dict=confusion_matrices,
    normalize=True,
    save_path=conf.OUTPUT_DIR / "confusion_matrices.png",
)

# Analisi qualitativa sul modello migliore tra i due
best_exp = max(test_results, key=lambda k: test_results[k][1]["miou"])
best_ckpt = conf.OUTPUT_DIR / f"best_{best_exp}.pth"
ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])

plot_qualitative_results(
    model=model,
    test_loader=test_loader,
    device=device,
    num_samples=4,
    save_path=conf.OUTPUT_DIR / f"qualitative_{best_exp}.png",
)
